# Mirai Milestone 3 — one-case PANORAMA GPU pilot

Prepared 2026-10-14. **This tracked notebook has not completed a real GPU run.**
Open in free Colab, choose **Runtime → Change runtime type → T4 GPU** (or a
better available NVIDIA GPU), then execute cells in order. Free allocation is
not guaranteed. No Docker, training, or installation on the Windows laptop.

Only upload one permitted CT export: `ct.mha`, `input-provenance.json`, and
`clinical-information-pancreatic-ct.json`. **Do not upload the annotation.**
The existing public MSD case is a software smoke test: training overlap,
venous-phase suitability, HU calibration and clinical metadata are unresolved.

The official final outputs are a candidate detection map and an uncalibrated
scalar score. Neither is a clinical risk probability or a Grad-CAM explanation.
No accuracy can be established from this case.

Sources: [official baseline](https://github.com/DIAGNijmegen/PANORAMA_baseline/tree/d08f2356fa70d9460881fec0aedba4ebd1566c7e),
[weights and published checksums](https://zenodo.org/records/11160381),
[build log](https://github.com/DIAGNijmegen/PANORAMA_baseline/blob/d08f2356fa70d9460881fec0aedba4ebd1566c7e/build.log),
[Colab resource limits](https://research.google.com/colaboratory/faq.html).

## Cell 1 — permission and configuration

Review the separate licenses: baseline/nnU-Net code Apache-2.0, weights
**CC BY-NC 4.0**, public MSD input **CC BY-SA 4.0**, lesion extraction code MIT.
Set `PERMITTED_NONCOMMERCIAL_USE = True` only for permitted student/noncommercial
use. Company commercial use requires separate permission. This is an operator
declaration, not a license determination by the notebook. Keep the smoke-test
flag True for our current sample; unknown clinical fields remain JSON nulls.

The source pins are immutable. The runner revision below contains the GPU
runner and exact dependency lock. The numerical dependencies follow the build
log; scikit-image 0.21.0 replaces 0.19.3 for a Python 3.11 wheel. Full inference
compatibility still requires this pilot. Both models use all five folds,
original TTA, original resampler/crop/postprocessing and saved nnU-Net plans.

In [13]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

PERMITTED_NONCOMMERCIAL_USE = True
SOFTWARE_SMOKE_TEST = True
MIRAI_REV = "1157be07c6954d1ec08f95a2b3bbf6d898f6b44d"
BASELINE_REV = "d08f2356fa70d9460881fec0aedba4ebd1566c7e"
ROOT = Path("/content/mirai-panorama")
PROJECT = ROOT / "mirai"
BASELINE = ROOT / "baseline"
INPUT = ROOT / "input"
PYTHON = ROOT / "gpu-env/bin/python"
if not sys.platform.startswith("linux") or not Path("/content").exists():
    raise RuntimeError("Run this notebook in a Colab NVIDIA GPU runtime; not on Windows.")
if not PERMITTED_NONCOMMERCIAL_USE:
    raise RuntimeError("Review the licenses, then set PERMITTED_NONCOMMERCIAL_USE=True.")
ROOT.mkdir(exist_ok=True)

## Cell 2 — resources before large downloads

The gates below are conservative pilot policy, **not measured minimum baseline
requirements**: free GPU memory ≥12 GiB, available host RAM ≥8 GiB, free disk
≥15 GiB. Actual peak use/runtime are unknown. If this gate fails, reconnect a
suitable runtime or use a company GPU with permitted weights. Do not reduce
folds or substitute a different model to pass the gate.

In [14]:
probe = (
    subprocess.check_output(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total,memory.free,driver_version",
            "--format=csv,noheader,nounits",
        ],
        text=True,
    )
    .strip()
    .splitlines()
)
if len(probe) != 1:
    raise RuntimeError("Select a runtime with one visible NVIDIA GPU.")
gpu, total_mib, free_mib, driver = [item.strip() for item in probe[0].split(",")]
meminfo = {
    line.split(":")[0]: int(line.split()[1]) * 1024
    for line in Path("/proc/meminfo").read_text().splitlines()
    if line.startswith(("MemTotal:", "MemAvailable:"))
}
disk_gib = shutil.disk_usage(ROOT).free / 1024**3
resource_check = dict(
    gpu=gpu,
    vram_total_mib=float(total_mib),
    vram_free_mib=float(free_mib),
    driver=driver,
    host_available_gib=meminfo["MemAvailable"] / 1024**3,
    disk_free_gib=disk_gib,
)
print(json.dumps(resource_check, indent=2))
if float(free_mib) < 12 * 1024 or meminfo["MemAvailable"] < 8 * 1024**3 or disk_gib < 15:
    raise RuntimeError("Resource gate failed; do not download models in this runtime.")

{
  "gpu": "Tesla T4",
  "vram_total_mib": 15360.0,
  "vram_free_mib": 14913.0,
  "driver": "580.82.07",
  "host_available_gib": 11.213016510009766,
  "disk_free_gib": 60.58778381347656
}


## Cell 3 — pinned source and isolated Python

Colab's default Python can change. Use uv 0.8.22 from PyPI to create a separate
Python 3.11 environment, without replacing the notebook kernel. The model runs
in subprocesses there. No Docker/privileged daemon is needed. Git LFS smudging
is disabled; weights come only from checksum-verified Zenodo archives later.

In [15]:
def checkout(url, destination, revision):
    env = dict(os.environ, GIT_LFS_SKIP_SMUDGE="1")
    if not destination.exists():
        subprocess.run(["git", "init", str(destination)], check=True, env=env)
        subprocess.run(
            ["git", "-C", str(destination), "remote", "add", "origin", url], check=True, env=env
        )
        subprocess.run(
            ["git", "-C", str(destination), "fetch", "--depth", "1", "origin", revision],
            check=True,
            env=env,
        )
        subprocess.run(
            ["git", "-C", str(destination), "checkout", "--detach", "FETCH_HEAD"],
            check=True,
            env=env,
        )
    actual = subprocess.check_output(
        ["git", "-C", str(destination), "rev-parse", "HEAD"], text=True
    ).strip()
    if actual != revision:
        raise RuntimeError("Existing checkout has a different revision; use a fresh runtime.")


checkout("https://github.com/MihirG1414/pancreatic_cancer_ai.git", PROJECT, MIRAI_REV)
checkout("https://github.com/DIAGNijmegen/PANORAMA_baseline.git", BASELINE, BASELINE_REV)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "uv==0.8.22"], check=True)
UV = shutil.which("uv")
if UV is None:
    raise RuntimeError("uv executable missing after installation.")
if not PYTHON.exists():
    subprocess.run([UV, "venv", "--seed", "--python", "3.11", str(ROOT / "gpu-env")], check=True)

## Cell 4 — install the resolved GPU dependencies

This downloads GPU packages to Colab only. The lock resolves Linux/Python 3.11
dependencies and pins the nnU-Net source revision. PyTorch 2.4.0 brings its CUDA
runtime libraries; the selected Colab driver must support them. Installation,
`pip check`, imports and actual CUDA availability must pass before weights.

In [16]:
subprocess.run(
    [
        UV,
        "pip",
        "install",
        "--python",
        str(PYTHON),
        "-r",
        str(PROJECT / "requirements-panorama-gpu-lock.txt"),
    ],
    check=True,
)
subprocess.run([str(PYTHON), "-m", "pip", "check"], check=True)
ENV = dict(
    os.environ,
    PYTHONPATH=str(PROJECT / "src"),
    PATH=str(PYTHON.parent) + os.pathsep + os.environ["PATH"],
)
subprocess.run(
    [
        str(PYTHON),
        "-c",
        "import torch; from mirai_ct.panorama_runtime import _versions; "
        "print(_versions()); print('CUDA:', torch.version.cuda); "
        "assert torch.cuda.is_available(), 'PyTorch cannot access GPU'; "
        "print(torch.cuda.get_device_name(0))",
    ],
    check=True,
    env=ENV,
)
subprocess.run(
    [str(PYTHON), "-m", "mirai_ct.panorama_runtime", "preflight", "--root", str(ROOT)],
    check=True,
    env=ENV,
)

CompletedProcess(args=['/content/mirai-panorama/gpu-env/bin/python', '-m', 'mirai_ct.panorama_runtime', 'preflight', '--root', '/content/mirai-panorama'], returncode=0)

## Cell 5 — upload one CT export, with no annotation

Select exactly the three files from your ignored Windows export directory.
The clinical JSON has the four required fields with null values for our sample.
The official baseline only prints these fields; this adaptation retains them
and avoids printing potentially identifying metadata. Nulls are accepted only
as an explicit software smoke test; they do not establish input suitability.

In [17]:
from google.colab import files

expected_names = {"ct.mha", "input-provenance.json", "clinical-information-pancreatic-ct.json"}
if not INPUT.exists():
    uploaded = files.upload()
    if set(uploaded) != expected_names:
        raise RuntimeError("Upload exactly the three CT export files; no annotations.")
    if sum(len(value) for value in uploaded.values()) > 1024**3:
        raise RuntimeError("Input exceeds one-case 1 GiB upload budget.")
    INPUT.mkdir()
    for name, value in uploaded.items():
        (INPUT / name).write_bytes(value)
    del uploaded
if {p.name for p in INPUT.iterdir()} != expected_names:
    raise RuntimeError("Input folder must contain exactly one exported CT and its two JSON files.")
info = json.loads((INPUT / "input-provenance.json").read_text())
clinical = json.loads((INPUT / "clinical-information-pancreatic-ct.json").read_text())
unknown = any(value is None for value in clinical.values()) or any(
    value == "unknown" for value in info["suitability"].values()
)
if unknown and not SOFTWARE_SMOKE_TEST:
    raise RuntimeError(
        "Unknown input suitability/clinical fields require SOFTWARE_SMOKE_TEST=True."
    )
print("CT only; suitability:", info["suitability"], "software smoke test:", SOFTWARE_SMOKE_TEST)
check_input = """
import json, sys
from pathlib import Path
from mirai_ct.panorama_io import sha256_file, read_aligned_mha
p = Path(sys.argv[1]); info = json.loads((p/'input-provenance.json').read_text())
assert info['annotation_used'] is False
assert sha256_file(p/'ct.mha') == info['mha_sha256']
image = read_aligned_mha(p/'ct.mha', info['geometry'])
print('Checked CT grid:', image.GetSize())
"""
subprocess.run([str(PYTHON), "-c", check_input, str(INPUT)], check=True, env=ENV)

CT only; suitability: {'contrast_phase': 'unknown', 'hu_calibration': 'unknown'} software smoke test: True


CompletedProcess(args=['/content/mirai-panorama/gpu-env/bin/python', '-c', "\nimport json, sys\nfrom pathlib import Path\nfrom mirai_ct.panorama_io import sha256_file, read_aligned_mha\np = Path(sys.argv[1]); info = json.loads((p/'input-provenance.json').read_text())\nassert info['annotation_used'] is False\nassert sha256_file(p/'ct.mha') == info['mha_sha256']\nimage = read_aligned_mha(p/'ct.mha', info['geometry'])\nprint('Checked CT grid:', image.GetSize())\n", '/content/mirai-panorama/input'], returncode=0)

## Cell 6 — official weights, checksums and all ten checkpoints

Approximately 1.64 GiB of compressed archives plus extracted models and scratch
space. Checksum/size verification happens before extraction and existing
archives are reused only after verification. Published MD5 detects corruption;
local SHA256 adds provenance. Missing folds fail. No single-fold shortcut.
Partial downloads fail explicitly; keep logs, remove only the indicated
`.part` in this Colab runtime and retry. A partially extracted model tree must
be repaired in a fresh runtime or verified manually before reuse.

In [18]:
subprocess.run(
    [
        str(PYTHON),
        "-m",
        "mirai_ct.panorama_runtime",
        "weights",
        "--root",
        str(ROOT),
        "--permitted-noncommercial-use",
    ],
    check=True,
    env=ENV,
)
print(json.loads((ROOT / "weight-provenance.json").read_text()))

{'record': 'https://zenodo.org/records/11160381', 'license': 'CC BY-NC 4.0', 'permission_confirmed_by_operator': True, 'archives': [{'filename': 'Dataset103_PANORAMA_baseline_Pancreas_Segmentation.zip', 'published_md5': 'd4c7e9666157e712f90649086ed395a5', 'size': 613484406, 'sha256': '8e1e97a3654f041029a5f08d86d8616d833346c20696b82d4b23f9ace83730b6'}, {'filename': 'Dataset104_PANORAMA_baseline_PDAC_Detection.zip', 'published_md5': 'ae1bc3ff68e4e0b9f7fa15afca339cf8', 'size': 1143995491, 'sha256': 'e83135b120970859579632c5692d3653282e8af5a9e94cc655f0df0ae654e5fd'}], 'license_sha256': '98679f478515c0f5c50e720006d12b4c3800d488e8dccca4e59a657efbb86c6e', 'model_sha256': {'Dataset103_PANORAMA_baseline_Pancreas_Segmentation/nnUNetTrainer__nnUNetPlans__3d_fullres/plans.json': 'cadc013b81fad128940b82fad7691fbf9b4167fffb1562b6b29ccda0773b02ac', 'Dataset103_PANORAMA_baseline_Pancreas_Segmentation/nnUNetTrainer__nnUNetPlans__3d_fullres/dataset.json': 'ad4dcf1e6e69fbe8f24fb140617d215772a1e7d9aadaeca

## Cell 7 — run both pretrained stages once

This can take a substantial amount of time. Logs are saved per stage. The runner
uses 4.5×4.5×9 mm resampling, predicted-pancreas crop margins 100/50/15 mm,
five pancreas folds and five PDAC folds, original TTA/postprocessing and full
grid restoration. One preprocessing/export worker bounds host use; the
baseline inference defaults are preserved. Empty pancreas, invalid crop,
missing checkpoints, subprocess errors or invalid/misaligned outputs fail.

Do not rerun into an existing run directory. A failed `run.json` records the
failure; choose a new directory for a reviewed retry. No failure output is
packaged as a successful prediction.

In [ ]:
RUN_DIR = ROOT / ("run-" + time.strftime("%Y%m%d-%H%M%S", time.gmtime()))
try:
    subprocess.run(
        [
            str(PYTHON),
            "-m",
            "mirai_ct.panorama_runtime",
            "run",
            "--root",
            str(ROOT),
            "--input",
            str(INPUT),
            "--baseline",
            str(BASELINE),
            "--run-dir",
            str(RUN_DIR),
        ],
        check=True,
        env=ENV,
    )
except subprocess.CalledProcessError:
    for log in sorted(RUN_DIR.glob("stage-*.log")):
        print(log.name, log.read_text(errors="replace")[-6000:])
    raise

In [25]:
ENV["MPLBACKEND"] = "Agg"
ENV["PYTHONDONTWRITEBYTECODE"] = "1"

subprocess.run(
    [
        str(PYTHON), "-c",
        "import matplotlib; "
        "print('Matplotlib backend:', matplotlib.get_backend())"
    ],
    env=ENV,
    check=True,
)

CompletedProcess(args=['/content/mirai-panorama/gpu-env/bin/python', '-c', "import matplotlib; print('Matplotlib backend:', matplotlib.get_backend())"], returncode=0)

In [20]:
from pathlib import Path
import json

failed_run = Path("/content/mirai-panorama/run-20261027-195113")
record_path = failed_run / "bundle/run.json"

print("Run folder exists:", failed_run.exists())

if record_path.is_file():
    record = json.loads(record_path.read_text())
    for key in ("status", "failure_type", "failure_message"):
        print(f"{key}: {record.get(key)}")
else:
    print("No run.json: failure occurred before run logging started.")

logs = sorted(failed_run.glob("stage-*.log"))
for log in logs:
    print(f"\n--- {log.name}: last 8,000 characters ---")
    print(log.read_text(errors="replace")[-8000:])

if not logs:
    print("No stage logs found.")

Run folder exists: False
No run.json: failure occurred before run logging started.
No stage logs found.


In [21]:
import os
import subprocess
from pathlib import Path
from uuid import uuid4

ROOT = Path("/content/mirai-panorama")
PROJECT = ROOT / "mirai"
BASELINE = ROOT / "baseline"
INPUT = ROOT / "input"
PYTHON = ROOT / "gpu-env/bin/python"

required = [
    PYTHON,
    PROJECT / "src/mirai_ct/panorama_runtime.py",
    BASELINE / "src/process.py",
    INPUT / "ct.mha",
    ROOT / "weight-provenance.json",
]
missing = [str(p) for p in required if not p.exists()]

if missing:
    print("Runtime files are missing:")
    print("\n".join(missing))
    print("Stop here; share this output.")
else:
    ENV = dict(
        os.environ,
        PYTHONPATH=str(PROJECT / "src"),
        PATH=str(PYTHON.parent) + os.pathsep + os.environ["PATH"],
    )
    RUN_DIR = ROOT / ("run-diagnostic-" + uuid4().hex[:12])

    result = subprocess.run(
        [
            str(PYTHON), "-m", "mirai_ct.panorama_runtime", "run",
            "--root", str(ROOT),
            "--input", str(INPUT),
            "--baseline", str(BASELINE),
            "--run-dir", str(RUN_DIR),
        ],
        env=ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    log_path = ROOT / (RUN_DIR.name + ".log")
    log_path.write_text(result.stdout)
    print("Exit code:", result.returncode)
    print("Run directory:", RUN_DIR)
    print(result.stdout[-16000:])

    for log in sorted(RUN_DIR.glob("stage-*.log")):
        print(f"\n--- {log.name} ---")
        print(log.read_text(errors="replace")[-8000:])

Exit code: 1
Run directory: /content/mirai-panorama/run-diagnostic-27a5f5fedf4e
GPU pilot failed: Baseline source contains local modifications; use the unmodified pin.



In [23]:
import os
import subprocess
from pathlib import Path

os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
ENV["PYTHONDONTWRITEBYTECODE"] = "1"

cache = Path(
    "/content/mirai-panorama/baseline/src/__pycache__/"
    "data_utils.cpython-311.pyc"
)
cache.unlink(missing_ok=True)

status = subprocess.check_output(
    ["git", "-C", str(BASELINE), "status", "--porcelain"],
    text=True,
)
print(status or "Baseline checkout is clean.")

Baseline checkout is clean.


## Cell 8 — verify genuine on-disk output and inspect a native CT slice

Only a successful completed run permits this cell. Sampled memory is not an
exact peak. The viewer uses the map maximum for slice selection; the annotation
never influenced preprocessing, crop or model inference.

In [22]:
import subprocess

for args in [
    ["rev-parse", "HEAD"],
    ["status", "--short"],
    ["diff", "--stat"],
    ["diff", "--cached", "--stat"],
]:
    print("\n git", " ".join(args))
    result = subprocess.run(
        ["git", "-C", "/content/mirai-panorama/baseline", *args],
        capture_output=True,
        text=True,
    )
    print(result.stdout or result.stderr or "(no changes)")


 git rev-parse HEAD
d08f2356fa70d9460881fec0aedba4ebd1566c7e


 git status --short
?? src/__pycache__/data_utils.cpython-311.pyc


 git diff --stat
(no changes)

 git diff --cached --stat
(no changes)


In [ ]:
from IPython.display import Image, display

BUNDLE = RUN_DIR / "bundle"
run = json.loads((BUNDLE / "run.json").read_text())
if run["status"] != "success" or run["execution_kind"] != "pretrained-inference":
    raise RuntimeError("No successful genuine model run is available.")
score = json.loads((BUNDLE / "pdac-likelihood.json").read_text())
print("Uncalibrated model score:", score)
print("Runtime seconds:", run["runtime_seconds"])
print("Sampled resources:", run["resource_samples"])
print("Outputs:", sorted(run["output_sha256"]))
plot_code = """
import sys, json, numpy as np, SimpleITK as sitk
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from mirai_ct.panorama_io import validate_alignment
p, b = map(Path, sys.argv[1:]); info=json.loads((p/'input-provenance.json').read_text())
ct=sitk.ReadImage(str(p/'ct.mha'))
det=sitk.ReadImage(str(b/'images/pdac-detection-map/detection_map.mha'))
validate_alignment(det, info['geometry'])
c=sitk.GetArrayViewFromImage(ct); d=sitk.GetArrayViewFromImage(det)
k=int(np.unravel_index(d.argmax(), d.shape)[0]) if d.max()>0 else d.shape[0]//2
fig,axes=plt.subplots(1,2,figsize=(12,5))
aspect=ct.GetSpacing()[1]/ct.GetSpacing()[0]
for ax in axes:
    ax.imshow(c[k], origin='lower', cmap='gray', vmin=-150, vmax=250,
              interpolation='nearest', aspect=aspect)
axes[0].set_title('CT: scaled intensities; HU unverified')
im=axes[1].imshow(np.ma.masked_less_equal(d[k],0), origin='lower', cmap='magma',
                  vmin=0, vmax=1, alpha=.65, interpolation='nearest', aspect=aspect)
axes[1].set_title('PANORAMA candidate detection map')
fig.colorbar(im,ax=axes[1],label='Uncalibrated candidate score')
for ax in axes: ax.set_xlabel('native i'); ax.set_ylabel('native j')
fig.suptitle(f'Native k={k}; original physical grid; no clinical performance claim')
fig.tight_layout(); fig.savefig(b/'gpu-preview.png',dpi=120); plt.close(fig)
"""
subprocess.run([str(PYTHON), "-c", plot_code, str(INPUT), str(BUNDLE)], check=True, env=ENV)
display(Image(filename=str(BUNDLE / "gpu-preview.png")))

## Cell 9 — download the output bundle

The ZIP includes detection map, scalar score, input provenance, run provenance,
model/archive hashes, dependency freeze, stage logs and weight license. It
contains no annotation or source CT. Download before disconnecting: free
Colab storage is temporary. On Windows, extract into a new ignored `outputs/`
directory, then follow `docs/milestone-3-demo.md` to validate/import and report.
The importer also requires your original verified input-export directory.

If a cell failed, report its traceback and saved logs. An unexecuted or failed
notebook is not a successful model run. Do not report one-case cancer accuracy.

In [ ]:
archive = RUN_DIR / "panorama-output.zip"
if run["status"] != "success" or not archive.is_file():
    raise RuntimeError("Successful output bundle missing.")
print("Download:", archive, "bytes:", archive.stat().st_size)
files.download(str(archive))